# GENZONE | AI Manthan - N-CMAPSS physics-informed, interpretable health & RUL training
Layers (same as the architecture diagram):
1. **Ingestion & preprocessing** - 1 Hz flight data -> one row per flight cycle; healthy baseline per engine
2. **Physics model** - first-principles ratios/efficiency proxies + healthy-engine map ("ideal" sensor values)
3. **Kalman filter** - estimates the 10 component health parameters (fan/LPC/HPC/HPT/LPT efficiency & flow)
4. **Interpretable ML** - XGBoost + LightGBM + CatBoost + Random Forest on per-cycle physics features, plus a 1D-CNN
   on 0.1 Hz in-flight windows enriched with the Layer 2/3 outputs (hybrid design of Arias Chao et al., RESS 2022);
   per target the non-negative Ridge blend is kept only if it beats the best single model in cross-validation;
   causal per-engine smoothing; targets: RUL and component health indices (fan, LPC, HPC, HPT, LPT); TreeSHAP
5/6. Exported bundle is served by the backend/dashboard

**How to run on Google Colab**
1. Runtime > Change runtime type > **T4 GPU** (CPU works too, the CNN is just slower). High-RAM helps for RUN="full".
2. Run all cells. Data is downloaded per file from Kaggle (`bishals098/nasa-cmapss-2-engine-degradation`).
   If Kaggle asks for credentials: add Colab secrets KAGGLE_USERNAME and KAGGLE_KEY (from kaggle.com > Settings >
   API > Create New Token), or put the .h5 files in Google Drive and set DATA_DIR.
3. The last cell downloads `genzone_ncmapss_<RUN>_bundle.zip`. Unzip it into `backend/models/` and start the backend.

**RUN = "quick"**: DS02 only (PHM 2021 challenge subset) - checks the pipeline end to end.
**RUN = "full"**: DS01-DS08 (10 files) combined - every component has abnormal degradation in at least one subset, which the
component-wise health indices need. Use this for the final model.

## 0. Setup

In [ ]:
import gc
import glob
import json
import os
import platform
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")
T_START = time.time()
TIMINGS = {}
if IN_COLAB or IN_KAGGLE:          # Kaggle images lack onnx/onnxruntime -> the CNN would silently miss the bundle
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "xgboost", "lightgbm", "catboost", "shap", "h5py",
                    "kagglehub", "onnx", "onnxruntime", "onnxscript"], check=False)
    try:    # make Colab secrets visible to kagglehub
        from google.colab import userdata
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                os.environ[k] = userdata.get(k)
            except Exception:
                pass
    except Exception:
        pass

## 1. Configuration (edit this cell)

In [ ]:
RUN = os.environ.get("RUN", "quick")                          # "quick" (DS02) or "full" (DS01-DS08d)
DATASET_HANDLE = "bishals098/nasa-cmapss-2-engine-degradation"
FILES = {"quick": ["N-CMAPSS_DS02-006.h5"],
         "full": ["N-CMAPSS_DS01-005.h5", "N-CMAPSS_DS02-006.h5", "N-CMAPSS_DS03-012.h5", "N-CMAPSS_DS04.h5",
                  "N-CMAPSS_DS05.h5", "N-CMAPSS_DS06.h5", "N-CMAPSS_DS07.h5", "N-CMAPSS_DS08a-009.h5",
                  "N-CMAPSS_DS08c-008.h5", "N-CMAPSS_DS08d-010.h5"]}[RUN]
DATA_DIR = os.environ.get("NCMAPSS_DIR")                      # e.g. "/content/drive/MyDrive/ncmapss" (optional)
DELETE_AFTER_USE = True                                       # delete each downloaded .h5 after aggregation (disk)
HEALTHY_SAMPLES_PER_FILE = 200_000                            # 1 Hz healthy rows per file for the healthy-engine map
BASELINE_CYCLES = 10                                          # healthy baseline window per engine (cycles)
ROLL = 5                                                      # rolling window for mean/slope features (cycles)
N_FOLDS = 5                                                   # GroupKFold by engine (capped at #engines)
SEED = 42
RUL_SCALE = 100.0
BAND_EDGES = [0.75, 0.55, 0.30]                               # Normal / Warning / Alarm / Emergency health bands
USE_CNN = True                                                # 1D-CNN on 0.1 Hz in-flight windows
CNN_EVERY = 10                                                # keep every 10th 1 Hz row -> 0.1 Hz
CNN_WINDOW = 50                                               # samples per window = 500 s of flight
CNN_EPOCHS = 20
CNN_BATCH = 1024
CNN_LR = 1e-3
CNN_SAMPLES_PER_EPOCH = 300_000                               # random windows drawn per epoch
STACK_MIN_GAIN = 0.03                                         # keep a blend only if >=3% better than best single (CV)
SMOOTH_GRID = [1.0, 0.7, 0.5, 0.35, 0.25, 0.15, 0.1]          # causal smoothing strengths tried (1 = off)
rng = np.random.default_rng(SEED)

## 2. Shared pipeline code (identical to backend/pipeline/ncmapss.py - do not edit here)

In [ ]:
"""NASA N-CMAPSS (C-MAPSS 2) digital-twin pipeline: Layers 1-3 plus feature engineering.

This ONE module is used by the Colab training notebook (inlined) and by the backend, so models are trained and
served on identical computations. Dependencies: numpy, pandas (+ h5py only for reading the raw .h5 files).

Layer 1  load_h5 / aggregate_cycles : 1 Hz flight data -> one row per (unit, flight cycle)
Layer 2  physics_1hz + HealthyReference : first-principles ratios/efficiency proxies, and a healthy-engine map
         (sensor = f(flight condition)) giving the "ideal" sensor value and the residual (actual vs ideal)
Layer 3  GasPathKF : Kalman filter on a linearised gas-path model, estimating the 10 component health
         parameters (efficiency/flow modifiers) with uncertainty, plus per-sensor innovation checks
Layer 4 inputs  CausalFeaturizer, build_windows : causal per-engine features for the trees;
         downsample_1hz + SampleEncoder : 0.1 Hz in-flight sequences (+ residuals, KF states) for the CNN
Post-processing  smooth_target : causal per-engine smoothing of predictions (RUL drops 1 per cycle)

Conventions: health index 1.0 = healthy, 0.0 = failure-level degradation. Everything is causal per engine
(uses only the current and earlier cycles), so the same code runs cycle by cycle in real time.
"""
from __future__ import annotations

import itertools

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------------------------- schema
W_VARS = ["alt", "Mach", "TRA", "T2"]
XS_VARS = ["T24", "T30", "T48", "T50", "P15", "P2", "P21", "P24", "Ps30", "P40", "P50", "Nf", "Nc", "Wf"]
XV_DISPLAY = ["T40", "P30", "P45"]   # virtual (unmeasured) sensors: kept for display only, never model inputs
T_VARS = ["fan_eff_mod", "fan_flow_mod", "LPC_eff_mod", "LPC_flow_mod", "HPC_eff_mod", "HPC_flow_mod",
          "HPT_eff_mod", "HPT_flow_mod", "LPT_eff_mod", "LPT_flow_mod"]
COMPONENTS = ["fan", "lpc", "hpc", "hpt", "lpt"]
COMP_THETA = {"fan": ("fan_eff_mod", "fan_flow_mod"), "lpc": ("LPC_eff_mod", "LPC_flow_mod"),
              "hpc": ("HPC_eff_mod", "HPC_flow_mod"), "hpt": ("HPT_eff_mod", "HPT_flow_mod"),
              "lpt": ("LPT_eff_mod", "LPT_flow_mod")}
COMP_NAMES = {"fan": "Fan", "lpc": "LPC", "hpc": "HPC", "hpt": "HPT", "lpt": "LPT"}

SENSOR_INFO = {   # description, kind ("T" degR, "P" psia, "N" rpm, "W" lbm/s)
    "T24": ("LPC outlet total temperature", "T"), "T30": ("HPC outlet total temperature", "T"),
    "T48": ("HPT outlet total temperature (EGT)", "T"), "T50": ("LPT outlet total temperature", "T"),
    "P15": ("Bypass duct total pressure", "P"), "P2": ("Fan inlet total pressure", "P"),
    "P21": ("Fan outlet total pressure", "P"), "P24": ("LPC outlet total pressure", "P"),
    "Ps30": ("HPC outlet static pressure", "P"), "P40": ("Burner outlet total pressure", "P"),
    "P50": ("LPT outlet total pressure", "P"), "Nf": ("Fan (LP spool) speed", "N"),
    "Nc": ("Core (HP spool) speed", "N"), "Wf": ("Fuel flow", "W"),
    "T40": ("Burner outlet temperature (virtual)", "T"), "P30": ("HPC outlet total pressure (virtual)", "P"),
    "P45": ("HPT outlet total pressure (virtual)", "P"),
}

GAMMA = 1.4
K_COLD = (GAMMA - 1.0) / GAMMA
GAMMA_HOT = 1.33
K_HOT = (GAMMA_HOT - 1.0) / GAMMA_HOT

# Physics proxies computed at 1 Hz from MEASURED signals only, and the sensors each one depends on
# (used by the occlusion explainer).
PROXY_DEPS = {
    "fan_pr": ["P21", "P2"], "lpc_pr": ["P24", "P21"], "hpc_pr": ["Ps30", "P24"],
    "eta_core": ["P24", "P2", "T24"], "eta_hpc": ["Ps30", "P24", "T30", "T24"],
    "burner_dp": ["P40", "Ps30"], "lpt_tr": ["T50", "T48"], "turb_pr": ["P40", "P50"],
    "nf_corr": ["Nf"], "nc_corr": ["Nc", "T24"], "wf_corr": ["Wf", "P2"], "spool_ratio": ["Nc", "Nf"],
}
PROXIES = list(PROXY_DEPS)

# Fixed (not data-fitted) centring/scaling of the flight condition, so per-cycle polynomial means can be
# accumulated before any model is fitted.
W_CENTER = {"alt": (20000.0, 15000.0), "Mach": (0.5, 0.25), "TRA": (60.0, 20.0), "T2": (500.0, 30.0)}

DEG_R_TO_K = 5.0 / 9.0
PSIA_TO_KPA = 6.894757
FT_TO_M = 0.3048
LBM_TO_KG = 0.45359237


# ----------------------------------------------------------------------------------------------- Layer 1
def h5_names(f, key: str) -> list[str]:
    return [str(s).strip() for s in np.array(f[key]).reshape(-1).astype("U32")]


def load_h5(path: str, split: str = "dev", virtual=XV_DISPLAY) -> pd.DataFrame:
    """Read one split ("dev" or "test") of an N-CMAPSS .h5 file into a 1 Hz DataFrame (float32).

    Columns are mapped by the variable names stored in the file (W_var, X_s_var, ...), not by position.
    """
    import h5py

    with h5py.File(path, "r") as f:
        frames = []
        for grp in ("W", "X_s", "T", "A"):
            names = h5_names(f, f"{grp}_var")
            dtype = np.int32 if grp == "A" else np.float32
            frames.append(pd.DataFrame(np.asarray(f[f"{grp}_{split}"], dtype=dtype), columns=names))
        frames.append(pd.DataFrame({"RUL": np.asarray(f[f"Y_{split}"], dtype=np.float32).reshape(-1)}))
        if virtual and f"X_v_{split}" in f:
            vnames = h5_names(f, "X_v_var")
            keep = [vnames.index(v) for v in virtual if v in vnames]
            if keep:
                arr = np.asarray(f[f"X_v_{split}"][:, sorted(keep)], dtype=np.float32)
                frames.append(pd.DataFrame(arr, columns=["v_" + vnames[i] for i in sorted(keep)]))
    df = pd.concat(frames, axis=1)
    missing = [c for c in W_VARS + XS_VARS + ["unit", "cycle"] if c not in df.columns]
    if missing:
        raise ValueError(f"{path} [{split}] is missing variables {missing}")
    # physically impossible rows (e.g. zero-filled bytes of a repaired truncated file): sensors are always > 0
    bad = (df[XS_VARS].to_numpy() <= 0).any(axis=1) | (df["unit"].to_numpy() < 1) | (df["cycle"].to_numpy() < 1)
    if bad.any():
        df = df.loc[~bad].reset_index(drop=True)
    df.attrs["dropped_rows"] = int(bad.sum())
    return df


def repair_truncated_h5(path: str, workdir: str, max_missing: int = 4096):
    """Some mirrors ship an .h5 a few bytes shorter than its header records (Kaggle's DS08d-010: 32 bytes), which
    HDF5 refuses to open. Copy it to `workdir` and zero-pad the copy up to the recorded size (load_h5 then drops any
    zeroed rows). Returns (path to open, repaired copy to delete afterwards or None)."""
    import os
    import re
    import shutil

    import h5py

    try:
        h5py.File(path, "r").close()
        return path, None
    except OSError as exc:
        m = re.search(r"eof = (\d+), sblock->base_addr = \d+, stored_eof = (\d+)", str(exc))
        if not m or not 0 < int(m.group(2)) - int(m.group(1)) <= max_missing:
            raise
        missing = int(m.group(2)) - int(m.group(1))
    fixed = os.path.join(workdir, "repaired_" + os.path.basename(path))
    shutil.copyfile(path, fixed)
    with open(fixed, "ab") as fh:
        fh.write(b"\0" * missing)
    return fixed, fixed


def physics_1hz(df: pd.DataFrame) -> pd.DataFrame:
    """First-principles ratios from measured signals (unit-free; temperatures in degR, pressures in psia).

    Efficiencies are isentropic PROXIES: eta_core spans fan-hub + LPC (T2,P2 -> T24,P24); eta_hpc uses the
    measured static Ps30 because total P30 is not measured.
    """
    def ratio(a, b):
        b = np.asarray(b, dtype=np.float64)
        return np.asarray(a, dtype=np.float64) / np.where(np.abs(b) < 1e-9, np.nan, b)

    def eta(pr, tr):
        pr = np.clip(pr, 1e-6, None)
        return np.clip(ratio(pr ** K_COLD - 1.0, tr - 1.0), 0.0, 1.5)

    out = pd.DataFrame(index=df.index)
    out["fan_pr"] = ratio(df["P21"], df["P2"])
    out["lpc_pr"] = ratio(df["P24"], df["P21"])
    out["hpc_pr"] = ratio(df["Ps30"], df["P24"])
    out["eta_core"] = eta(ratio(df["P24"], df["P2"]), ratio(df["T24"], df["T2"]))
    out["eta_hpc"] = eta(out["hpc_pr"].to_numpy(), ratio(df["T30"], df["T24"]))
    out["burner_dp"] = 1.0 - ratio(df["P40"], df["Ps30"])
    out["lpt_tr"] = ratio(df["T50"], df["T48"])
    out["turb_pr"] = ratio(df["P40"], df["P50"])
    theta2 = ratio(df["T2"], 518.67)
    delta2 = ratio(df["P2"], 14.696)
    out["nf_corr"] = ratio(df["Nf"], np.sqrt(theta2))
    out["nc_corr"] = ratio(df["Nc"], np.sqrt(ratio(df["T24"], 518.67)))
    out["wf_corr"] = ratio(df["Wf"], delta2 * np.sqrt(theta2))
    out["spool_ratio"] = ratio(df["Nc"], df["Nf"])
    return out.astype(np.float32)


def scaled_w(w: np.ndarray) -> np.ndarray:
    c = np.array([W_CENTER[v][0] for v in W_VARS], dtype=np.float64)
    s = np.array([W_CENTER[v][1] for v in W_VARS], dtype=np.float64)
    return (np.asarray(w, dtype=np.float64) - c) / s


def poly_terms(n_vars: int, degree: int) -> list[tuple]:
    terms = [()]
    for d in range(1, degree + 1):
        terms += list(itertools.combinations_with_replacement(range(n_vars), d))
    return terms


def poly_features(z: np.ndarray, degree: int = 3) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64)
    cols = [np.prod(z[:, list(t)], axis=1) if t else np.ones(len(z)) for t in poly_terms(z.shape[1], degree)]
    return np.column_stack(cols)


def aggregate_cycles(df: pd.DataFrame, degree: int = 3) -> pd.DataFrame:
    """Collapse 1 Hz rows into one row per (unit, cycle).

    Stores per-cycle means of sensors (m_*), their std (sd_*), physics proxies (p_*), virtual sensors (v_*),
    flight descriptors, labels (RUL, health parameters, hs) and the mean polynomial features of the flight
    condition (phi_*). Because the healthy reference is linear in phi, mean(reference) = coef . mean(phi),
    which gives exact per-cycle "ideal" sensor means without a second pass over the 1 Hz data.
    """
    out = []
    vcols = [c for c in df.columns if c.startswith("v_")]
    tcols = [c for c in T_VARS if c in df.columns]
    for unit, part in df.groupby("unit", sort=True):
        _, inv = np.unique(part["cycle"].to_numpy(), return_inverse=True)
        cnt = np.bincount(inv).astype(np.float64)

        def gmean(a):
            """NaN-aware per-cycle mean."""
            a = np.asarray(a, dtype=np.float64)
            a = a.reshape(len(a), -1)
            ok = np.isfinite(a)
            sums = np.column_stack([np.bincount(inv, weights=np.where(ok[:, j], a[:, j], 0.0), minlength=len(cnt))
                                    for j in range(a.shape[1])])
            n = np.column_stack([np.bincount(inv, weights=ok[:, j].astype(float), minlength=len(cnt))
                                 for j in range(a.shape[1])])
            return sums / np.where(n == 0, np.nan, n)

        def gmax(a):
            res = np.full(len(cnt), -np.inf)
            np.maximum.at(res, inv, np.asarray(a, dtype=np.float64))
            return res

        def gmin(a):
            res = np.full(len(cnt), np.inf)
            np.minimum.at(res, inv, np.asarray(a, dtype=np.float64))
            return res

        rows = {"unit": np.full(len(cnt), int(unit)), "cycle": gmin(part["cycle"]).astype(int),
                "duration_s": cnt}
        if "Fc" in part:
            rows["fc"] = gmin(part["Fc"]).astype(int)
        if "hs" in part:
            rows["hs"] = gmin(part["hs"]).astype(int)
        if "RUL" in part:
            rows["rul"] = gmin(part["RUL"])
        w = part[W_VARS].to_numpy(dtype=np.float64)
        wm = gmean(w)
        for j, v in enumerate(W_VARS):
            rows[f"{v}_mean"] = wm[:, j]
        rows["alt_max"] = gmax(part["alt"])
        rows["Mach_max"] = gmax(part["Mach"])
        x = part[XS_VARS].to_numpy(dtype=np.float64)
        xm, x2 = gmean(x), gmean(x ** 2)
        for j, s in enumerate(XS_VARS):
            rows[f"m_{s}"] = xm[:, j]
            rows[f"sd_{s}"] = np.sqrt(np.clip(x2[:, j] - xm[:, j] ** 2, 0, None))
        pm = gmean(physics_1hz(part).to_numpy())
        for j, p in enumerate(PROXIES):
            rows[f"p_{p}"] = pm[:, j]
        if vcols:
            vm = gmean(part[vcols].to_numpy())
            for j, c in enumerate(vcols):
                rows[c] = vm[:, j]
        if tcols:
            tm = gmean(part[tcols].to_numpy())
            for j, c in enumerate(tcols):
                rows[c] = tm[:, j]
        phim = gmean(poly_features(scaled_w(w), degree))
        for j in range(phim.shape[1]):
            rows[f"phi_{j}"] = phim[:, j]
        out.append(pd.DataFrame(rows))
    return pd.concat(out, ignore_index=True).sort_values(["unit", "cycle"]).reset_index(drop=True)


# ----------------------------------------------------------------------------------------------- Layer 2
class HealthyReference:
    """Healthy-engine map: each measured sensor as a degree-3 polynomial of the flight condition (alt, Mach,
    TRA, T2), fitted by ridge regression on healthy (hs == 1) flight data of the training engines only.

    It plays the role of the diagram's "ideal performance" model: for any flight condition it predicts what a
    healthy engine would read, so residual = actual - ideal isolates degradation from operating conditions.
    """

    def __init__(self, degree: int = 3, alpha: float = 1e-3, sensors=XS_VARS):
        self.degree, self.alpha, self.sensors = degree, alpha, list(sensors)
        self.coef = None            # (n_terms, n_sensors)
        self.fit_rmse = None

    def fit(self, w: np.ndarray, x: np.ndarray) -> "HealthyReference":
        phi = poly_features(scaled_w(w), self.degree)
        a = phi.T @ phi + self.alpha * np.eye(phi.shape[1])
        self.coef = np.linalg.solve(a, phi.T @ np.asarray(x, dtype=np.float64))
        pred = phi @ self.coef
        self.fit_rmse = (np.sqrt(np.mean((pred - x) ** 2, axis=0)) / np.abs(np.mean(x, axis=0))).tolist()
        return self

    def expected_from_phi(self, phi_mean: np.ndarray) -> np.ndarray:
        return np.asarray(phi_mean, dtype=np.float64) @ self.coef

    def predict(self, w: np.ndarray) -> np.ndarray:
        return poly_features(scaled_w(w), self.degree) @ self.coef

    def to_dict(self):
        return {"degree": self.degree, "alpha": self.alpha, "sensors": self.sensors,
                "coef": np.asarray(self.coef).tolist(), "fit_rel_rmse": self.fit_rmse, "w_center": W_CENTER}

    @classmethod
    def from_dict(cls, d):
        r = cls(d["degree"], d["alpha"], d["sensors"])
        r.coef = np.asarray(d["coef"], dtype=np.float64)
        r.fit_rmse = d.get("fit_rel_rmse")
        return r


def add_residuals(cyc: pd.DataFrame, ref: HealthyReference) -> pd.DataFrame:
    """Add ref_<sensor> (ideal, healthy-engine value) and res_<sensor> (relative residual, actual/ideal - 1)."""
    cyc = cyc.copy()
    phi_cols = [f"phi_{j}" for j in range(ref.coef.shape[0])]
    exp = ref.expected_from_phi(cyc[phi_cols].to_numpy())
    for j, s in enumerate(ref.sensors):
        cyc[f"ref_{s}"] = exp[:, j]
        cyc[f"res_{s}"] = cyc[f"m_{s}"] / np.where(np.abs(exp[:, j]) < 1e-9, np.nan, exp[:, j]) - 1.0
    return cyc


# ----------------------------------------------------------------------------------------------- labels
def component_damage(df: pd.DataFrame) -> pd.DataFrame:
    """Per-component damage = larger of |efficiency modifier| and |flow modifier|."""
    return pd.DataFrame({c: np.maximum(df[e].abs(), df[f].abs()) for c, (e, f) in COMP_THETA.items()},
                        index=df.index)


def fit_damage_reference(cyc_train: pd.DataFrame, q: float = 0.95, floor: float = 1e-3) -> dict:
    """Failure-level damage PER COMPONENT: q-quantile over training engines of that component's damage at end of
    life. One shared value would let the component with the largest modifiers (e.g. fan flow) squeeze the others,
    so a turbine at end of life would still read ~0.8 health."""
    last = cyc_train.sort_values("cycle").groupby("unit").tail(1)
    dmg = component_damage(last)
    return {c: max(float(np.quantile(dmg[c], q)), floor) for c in COMPONENTS}


def d_ref_of(d_ref, comp: str) -> float:
    """d_ref may be one float (older bundles) or {component: float}."""
    return float(d_ref[comp]) if isinstance(d_ref, dict) else float(d_ref)


def component_health(df: pd.DataFrame, d_ref) -> pd.DataFrame:
    """HI_c = 1 - damage_c / d_ref_c, clipped to [0, 1]: fraction of failure-level degradation still unused."""
    dmg = component_damage(df)
    return pd.DataFrame({f"hi_{c}": (1.0 - dmg[c] / d_ref_of(d_ref, c)).clip(0.0, 1.0) for c in COMPONENTS},
                        index=df.index)


# ----------------------------------------------------------------------------------------------- Layer 3
SCHED_VARS = ["alt_mean", "Mach_mean", "TRA_mean"]


def schedule(cyc: pd.DataFrame) -> np.ndarray:
    """Operating-point variables for the influence coefficients: [1, alt, Mach, TRA] (fixed scaling)."""
    z = [np.ones(len(cyc))]
    for v, raw in zip(SCHED_VARS, ["alt", "Mach", "TRA"]):
        c, s = W_CENTER[raw]
        z.append((cyc[v].to_numpy(dtype=np.float64) - c) / s)
    return np.column_stack(z)


class GasPathKF:
    """Kalman filter on a linearised gas-path model (classic gas-path analysis).

    Measurement model per cycle:  res = b(w) + H(w) . theta + noise, where res are the 14 relative sensor
    residuals, theta the 10 component health parameters and w the operating point. H(w) (influence
    coefficients) is linear in [1, alt, Mach, TRA], i.e. the model is re-linearised at every operating point;
    because it is linear in theta at each point, the EKF update reduces to a time-varying Kalman filter.
    H and b are identified on training engines (where theta is known); at run time only sensors are used.
    State model: random walk with process noise Q estimated from cycle-to-cycle theta changes.
    """

    def __init__(self, sensors=XS_VARS, theta=T_VARS, ridge: float = 1e-4, gate: float = 9.0, q_scale: float = 1.0):
        self.sensors, self.theta = list(sensors), list(theta)
        self.ridge, self.gate, self.q_scale = ridge, gate, q_scale
        self.b = self.H = self.R = self.Q = self.x0 = self.P0 = None

    def _design(self, z, th):
        return np.column_stack([z] + [th[:, [p]] * z for p in range(th.shape[1])])

    def fit(self, cyc: pd.DataFrame) -> "GasPathKF":
        y = cyc[[f"res_{s}" for s in self.sensors]].to_numpy(dtype=np.float64)
        th = cyc[self.theta].to_numpy(dtype=np.float64)
        z = schedule(cyc)
        x = self._design(z, th)
        coef = np.linalg.solve(x.T @ x + self.ridge * np.eye(x.shape[1]), x.T @ y)    # (K + P*K, S)
        k = z.shape[1]
        self.b = coef[:k]                                                             # (K, S)
        self.H = coef[k:].reshape(len(self.theta), k, len(self.sensors))              # (P, K, S)
        self.R = np.var(y - x @ coef, axis=0) + 1e-12
        cyc_sorted = cyc.sort_values(["unit", "cycle"])
        d = cyc_sorted.groupby("unit")[self.theta].diff().dropna().to_numpy()
        self.Q = self.q_scale * (np.var(d, axis=0) + np.mean(d, axis=0) ** 2) + 1e-12
        first = cyc_sorted.groupby("unit")[self.theta].first().to_numpy()
        self.x0 = first.mean(axis=0)
        self.P0 = first.var(axis=0) + 1e-8
        return self

    def h_at(self, z_row):
        return np.einsum("pks,k->sp", self.H, z_row), z_row @ self.b                  # (S, P), (S,)

    def run(self, cyc: pd.DataFrame, res_override: np.ndarray | None = None) -> pd.DataFrame:
        """Filter every engine causally (sorted by unit, cycle). Returns states, 1-sigma, NIS and gate flags."""
        cyc = cyc.reset_index(drop=True)
        res = cyc[[f"res_{s}" for s in self.sensors]].to_numpy(dtype=np.float64) if res_override is None \
            else res_override
        z = schedule(cyc)
        n, p, s = len(cyc), len(self.theta), len(self.sensors)
        xs, sig = np.zeros((n, p)), np.zeros((n, p))
        nis, gated = np.full((n, s), np.nan), np.zeros((n, s), dtype=bool)
        units = cyc["unit"].to_numpy()
        for u in pd.unique(units):
            idx = np.flatnonzero(units == u)
            x, pm = self.x0.copy(), np.diag(self.P0)
            consec = np.zeros(s, dtype=int)
            for i in idx:
                pm = pm + np.diag(self.Q)
                h, b = self.h_at(z[i])
                for j in range(s):
                    r = res[i, j]
                    if not np.isfinite(r):
                        continue
                    hj = h[j]
                    innov = r - b[j] - hj @ x
                    ph = pm @ hj
                    sv = hj @ ph + self.R[j]
                    nis[i, j] = innov ** 2 / sv
                    if nis[i, j] > self.gate and consec[j] < 3:
                        gated[i, j] = True
                        consec[j] += 1
                        continue
                    consec[j] = 0
                    kg = ph / sv
                    x = x + kg * innov
                    ikh = np.eye(p) - np.outer(kg, hj)
                    pm = ikh @ pm @ ikh.T + self.R[j] * np.outer(kg, kg)           # Joseph form
                xs[i], sig[i] = x, np.sqrt(np.clip(np.diag(pm), 0, None))
        out = {}
        for k_, t in enumerate(self.theta):
            out[f"kf_{t}"] = xs[:, k_]
            out[f"kf_{t}_sigma"] = sig[:, k_]
        for j, sname in enumerate(self.sensors):
            out[f"nis_{sname}"] = nis[:, j]
            out[f"gated_{sname}"] = gated[:, j]
        return pd.DataFrame(out)

    def to_dict(self):
        return {"sensors": self.sensors, "theta": self.theta, "ridge": self.ridge, "gate": self.gate,
                "q_scale": self.q_scale, "sched": SCHED_VARS, "b": self.b.tolist(), "H": self.H.tolist(),
                "R": self.R.tolist(), "Q": self.Q.tolist(), "x0": self.x0.tolist(), "P0": self.P0.tolist()}

    @classmethod
    def from_dict(cls, d):
        k = cls(d["sensors"], d["theta"], d["ridge"], d["gate"], d["q_scale"])
        for a in ("b", "H", "R", "Q", "x0", "P0"):
            setattr(k, a, np.asarray(d[a], dtype=np.float64))
        return k


def kf_component_health(kf_out: pd.DataFrame, d_ref) -> pd.DataFrame:
    """Component HI from the filtered health parameters, with a 1-sigma from the dominating parameter."""
    out = {}
    for c, (e, f) in COMP_THETA.items():
        d = d_ref_of(d_ref, c)
        ae, af = kf_out[f"kf_{e}"].abs(), kf_out[f"kf_{f}"].abs()
        use_e = ae >= af
        out[f"kf_hi_{c}"] = (1.0 - np.maximum(ae, af) / d).clip(0, 1)
        out[f"kf_hi_{c}_sigma"] = np.where(use_e, kf_out[f"kf_{e}_sigma"], kf_out[f"kf_{f}_sigma"]) / d
    return pd.DataFrame(out, index=kf_out.index)


# ----------------------------------------------------------------------------------------------- Layer 4 inputs
FLIGHT_COLS = ["alt_mean", "Mach_mean", "TRA_mean", "T2_mean", "duration_s", "fc"]


def default_base_columns() -> list[str]:
    return [f"res_{s}" for s in XS_VARS] + [f"p_{p}" for p in PROXIES] + [f"kf_{t}" for t in T_VARS]


def source_of(feature: str) -> str:
    """Physical source of a model feature (for grouping SHAP values): sensor, proxy, KF state or flight."""
    base = feature.split("__")[0]
    for pre in ("res_", "p_", "kf_"):
        if base.startswith(pre):
            return base
    return base


class CausalFeaturizer:
    """Per-engine causal features: z-score, baseline-relative (vs median of the first `baseline_cycles`
    cycles of the same engine), rolling mean and rolling slope; plus flight descriptors and cycle count."""

    def __init__(self, base_cols=None, baseline_cycles: int = 10, roll: int = 5, use_cycle: bool = True):
        self.base_cols = list(base_cols) if base_cols else default_base_columns()
        self.flight_cols = list(FLIGHT_COLS)
        self.baseline_cycles, self.roll, self.use_cycle = baseline_cycles, roll, use_cycle
        self.mu = self.sd = None
        self.feature_names: list[str] = []
        self.seq_names: list[str] = []

    def fit(self, cyc: pd.DataFrame) -> "CausalFeaturizer":
        cols = self.base_cols + self.flight_cols
        self.mu = cyc[cols].mean().to_numpy(dtype=np.float64)
        sd = cyc[cols].std(ddof=0).to_numpy(dtype=np.float64)
        self.sd = np.where(sd < 1e-12, 1.0, sd)
        self._names()
        return self

    def _names(self):
        self.feature_names = []
        for c in self.base_cols:
            self.feature_names += [c, f"{c}__rel", f"{c}__rm", f"{c}__slope"]
        self.feature_names += self.flight_cols
        if self.use_cycle:
            self.feature_names.append("cycle")
        self.seq_names = self.base_cols + [f"{c}__rel" for c in self.base_cols] + self.flight_cols + \
            (["cycle_scaled"] if self.use_cycle else [])

    def transform(self, cyc: pd.DataFrame) -> pd.DataFrame:
        cyc = cyc.sort_values(["unit", "cycle"]).reset_index(drop=True)
        cols = self.base_cols + self.flight_cols
        zall = (cyc[cols].to_numpy(dtype=np.float64) - self.mu) / self.sd
        zall = np.nan_to_num(zall)
        nb = len(self.base_cols)
        z, zf = zall[:, :nb], zall[:, nb:]
        units = cyc["unit"].to_numpy()
        rel = np.empty_like(z)
        b = self.baseline_cycles
        for u in pd.unique(units):
            idx = np.flatnonzero(units == u)
            g = z[idx]
            base = np.empty_like(g)
            for i in range(min(len(g), b)):
                base[i] = np.median(g[: i + 1], axis=0)
            if len(g) > b:
                base[b:] = base[b - 1]
            rel[idx] = g - base
        reldf = pd.DataFrame(rel, columns=self.base_cols)
        rm = reldf.groupby(units).transform(lambda s_: s_.rolling(self.roll, min_periods=1).mean())
        slope = ((rm - rm.groupby(units).shift(self.roll)) / self.roll).fillna(0.0)
        out = {"unit": units, "cycle_id": cyc["cycle"].to_numpy()}
        for j, c in enumerate(self.base_cols):
            out[c] = z[:, j]
            out[f"{c}__rel"] = rel[:, j]
            out[f"{c}__rm"] = rm[c].to_numpy()
            out[f"{c}__slope"] = slope[c].to_numpy()
        for j, c in enumerate(self.flight_cols):
            out[c] = zf[:, j]
        if self.use_cycle:
            out["cycle"] = cyc["cycle"].to_numpy(dtype=np.float64)
            out["cycle_scaled"] = out["cycle"] / 100.0
        return pd.DataFrame(out)

    def to_dict(self):
        return {"base_cols": self.base_cols, "flight_cols": self.flight_cols, "baseline_cycles": self.baseline_cycles,
                "roll": self.roll, "use_cycle": self.use_cycle, "mu": self.mu.tolist(), "sd": self.sd.tolist(),
                "feature_names": self.feature_names, "seq_names": self.seq_names}

    @classmethod
    def from_dict(cls, d):
        f = cls(d["base_cols"], d["baseline_cycles"], d["roll"], d["use_cycle"])
        f.flight_cols = d["flight_cols"]
        f.mu, f.sd = np.asarray(d["mu"]), np.asarray(d["sd"])
        f._names()
        return f


def build_windows(frame: pd.DataFrame, cols: list[str], window: int):
    """Causal windows over cycles: window i = the `window` cycles ending at row i of the same engine (start of
    life padded by repeating the first cycle). Returns (big, starts); window i = big[starts[i]:starts[i]+window]."""
    return build_windows_array(frame[cols].to_numpy(np.float32), frame["unit"].to_numpy(), window)


def build_windows_array(seq: np.ndarray, units: np.ndarray, window: int):
    """build_windows on an array whose rows are grouped by engine and in time order within each engine."""
    starts = np.empty(len(seq), dtype=np.int64)
    chunks, offset = [], 0
    for u in pd.unique(units):
        idx = np.flatnonzero(units == u)
        g = seq[idx]
        padded = np.concatenate([np.repeat(g[:1], window - 1, axis=0), g], axis=0)
        starts[idx] = offset + np.arange(len(g))
        chunks.append(padded)
        offset += len(padded)
    return np.concatenate(chunks, axis=0), starts


def gather_windows(big: np.ndarray, starts: np.ndarray, window: int, rows) -> np.ndarray:
    rows = np.asarray(rows)
    return big[starts[rows][:, None] + np.arange(window)[None, :]]


def run_layers_1_to_3(cyc: pd.DataFrame, ref: HealthyReference, kf: GasPathKF, d_ref) -> pd.DataFrame:
    """Per-cycle table -> residuals (Layer 2) -> Kalman filter states and component HI (Layer 3)."""
    cyc = add_residuals(cyc.sort_values(["unit", "cycle"]).reset_index(drop=True), ref)
    k = kf.run(cyc)
    return pd.concat([cyc, k, kf_component_health(k, d_ref)], axis=1)


# ----------------------------------------------------------------------------------------------- post-processing
def clip_target(tgt: str, v) -> np.ndarray:
    return np.clip(v, 0, None) if tgt == "rul" else np.clip(v, 0, 1)


def causal_smooth(units, cycles, values, alpha: float, kind: str = "hi") -> np.ndarray:
    """Per-engine causal exponential smoothing of per-cycle predictions (rows sorted by unit, cycle; alpha=1: off).
    For RUL the previous estimate is first propagated by the known dynamics: RUL drops by 1 per flight cycle."""
    v = np.asarray(values, dtype=np.float64).copy()
    if alpha >= 1.0 or len(v) == 0:
        return v
    units, cycles = np.asarray(units), np.asarray(cycles, dtype=np.float64)
    for u in pd.unique(units):
        idx = np.flatnonzero(units == u)
        est = v[idx[0]]
        for k in range(1, len(idx)):
            prior = est - (cycles[idx[k]] - cycles[idx[k - 1]]) if kind == "rul" else est
            est = alpha * v[idx[k]] + (1.0 - alpha) * prior
            v[idx[k]] = est
    return v


def smooth_target(tgt: str, units, cycles, values, alpha: float) -> np.ndarray:
    return clip_target(tgt, causal_smooth(units, cycles, clip_target(tgt, values), alpha,
                                          "rul" if tgt == "rul" else "hi"))


# ----------------------------------------------------------------------------------------------- 0.1 Hz CNN inputs
SEQ_EVERY = 10          # keep every 10th 1 Hz row -> 0.1 Hz, as in Arias Chao et al. (hybrid physics + CNN)


def downsample_1hz(df: pd.DataFrame, every: int = SEQ_EVERY) -> pd.DataFrame:
    """Every `every`-th row of each (unit, cycle), keeping time order: unit, cycle, W, X_s (float32)."""
    pos = df.groupby(["unit", "cycle"], sort=False).cumcount().to_numpy()
    out = df.loc[pos % every == 0, ["unit", "cycle"] + W_VARS + XS_VARS].reset_index(drop=True)
    out[W_VARS + XS_VARS] = out[W_VARS + XS_VARS].astype(np.float32)
    out["unit"], out["cycle"] = out["unit"].astype(np.int64), out["cycle"].astype(np.int64)
    return out


def cycle_rows(samples: pd.DataFrame, cyc: pd.DataFrame) -> np.ndarray:
    """Row of `cyc` holding each sample's (unit, cycle), -1 if absent."""
    key = lambda d: d["unit"].to_numpy(np.int64) * 1_000_000 + d["cycle"].to_numpy(np.int64)   # noqa: E731
    return pd.Index(key(cyc)).get_indexer(key(samples))


class SampleEncoder:
    """CNN input channels for each 0.1 Hz sample: scaled flight condition, raw sensors, residual vs the healthy-engine
    map (Layer 2) and the engine's Kalman-filter health parameters for that cycle (Layer 3), z-scored on training
    samples. `occlude` replaces one sensor by its healthy-engine value (residual 0) for the occlusion explainer."""

    def __init__(self, sensors=XS_VARS, theta=T_VARS):
        self.sensors, self.theta = list(sensors), list(theta)
        self.mu = self.sd = None

    @property
    def channels(self) -> list[str]:
        return [f"w_{v}" for v in W_VARS] + [f"x_{s}" for s in self.sensors] + [f"res_{s}" for s in self.sensors] + \
            [f"kf_{t}" for t in self.theta]

    def raw(self, samples: pd.DataFrame, cyc: pd.DataFrame, ref: HealthyReference, occlude: str | None = None,
            chunk: int = 500_000):
        """-> (channels float32 (n, C), units (n,), cyc rows (n,)) for the samples whose cycle is in `cyc`."""
        rows = cycle_rows(samples, cyc)
        keep = np.flatnonzero(rows >= 0)
        rows = rows[keep]
        w_all = samples[W_VARS].to_numpy(np.float32)
        x_all = samples[self.sensors].to_numpy(np.float32)
        kf = cyc[[f"kf_{t}" for t in self.theta]].to_numpy(np.float64)
        n_w, n_s = len(W_VARS), len(self.sensors)
        out = np.empty((len(keep), len(self.channels)), dtype=np.float32)
        for i in range(0, len(keep), chunk):
            sel = keep[i:i + chunk]
            w, x = w_all[sel].astype(np.float64), x_all[sel].astype(np.float64)
            ideal = ref.predict(w)
            res = x / np.where(np.abs(ideal) < 1e-9, np.nan, ideal) - 1.0
            if occlude is not None:
                j = self.sensors.index(occlude)
                x[:, j], res[:, j] = ideal[:, j], 0.0
            out[i:i + chunk] = np.column_stack([scaled_w(w), x, np.nan_to_num(res), kf[rows[i:i + chunk]]])
        return out, samples["unit"].to_numpy()[keep], rows

    def fit(self, raw: np.ndarray, mask: np.ndarray, chunk: int = 500_000) -> "SampleEncoder":
        s = np.zeros(raw.shape[1])
        s2 = np.zeros(raw.shape[1])
        n = 0
        for i in range(0, len(raw), chunk):
            r = raw[i:i + chunk][mask[i:i + chunk]].astype(np.float64)
            s, s2, n = s + r.sum(0), s2 + (r ** 2).sum(0), n + len(r)
        self.mu = s / n
        sd = np.sqrt(np.clip(s2 / n - self.mu ** 2, 0, None))
        self.sd = np.where(sd < 1e-12, 1.0, sd)
        return self

    def normalize(self, raw: np.ndarray) -> np.ndarray:
        raw -= self.mu.astype(np.float32)
        raw /= self.sd.astype(np.float32)
        return raw

    def transform(self, samples, cyc, ref, occlude=None):
        raw, units, rows = self.raw(samples, cyc, ref, occlude)
        return self.normalize(raw), units, rows

    def to_dict(self):
        return {"sensors": self.sensors, "theta": self.theta, "channels": self.channels,
                "mu": self.mu.tolist(), "sd": self.sd.tolist()}

    @classmethod
    def from_dict(cls, d):
        e = cls(d["sensors"], d["theta"])
        e.mu, e.sd = np.asarray(d["mu"], dtype=np.float64), np.asarray(d["sd"], dtype=np.float64)
        return e


def cycle_mean(rows: np.ndarray, values: np.ndarray, n_rows: int) -> np.ndarray:
    """Average per-sample predictions (n, k) into per-cycle rows (n_rows, k); NaN for cycles without samples."""
    values = np.asarray(values, dtype=np.float64).reshape(len(rows), -1)
    cnt = np.bincount(rows, minlength=n_rows).astype(np.float64)
    sums = np.column_stack([np.bincount(rows, weights=values[:, j], minlength=n_rows) for j in range(values.shape[1])])
    return sums / np.where(cnt == 0, np.nan, cnt)[:, None]


def cnn_cycle_predictions(predict_windows, enc: SampleEncoder, samples: pd.DataFrame, cyc: pd.DataFrame,
                          ref: HealthyReference, window: int, occlude: str | None = None, batch: int = 8192):
    """Per-cycle CNN outputs (len(cyc), k): mean over the 0.1 Hz windows ending inside each flight cycle.
    `predict_windows` maps float32 (b, window, C) -> (b, k) in target units."""
    ch, units, rows = enc.transform(samples, cyc, ref, occlude)
    if len(ch) == 0:
        return None
    big, starts = build_windows_array(ch, units, window)
    del ch
    out = [predict_windows(gather_windows(big, starts, window, np.arange(i, min(i + batch, len(rows)))))
           for i in range(0, len(rows), batch)]
    return cycle_mean(rows, np.concatenate(out), len(cyc))

## 3. Download + Layer 1: aggregate 1 Hz flight data into one row per flight cycle (file by file)

In [ ]:
if "nc" not in globals():          # in the notebook the pipeline cell defines everything at top level
    import types
    nc = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith("_")})


def get_file(fname):
    for root in [r for r in (DATA_DIR, "/kaggle/input", "/content", "/content/drive/MyDrive", os.getcwd()) if r]:
        hits = glob.glob(os.path.join(root, "**", fname), recursive=True)
        if hits:
            return hits[0], False
    import kagglehub
    try:
        return kagglehub.dataset_download(DATASET_HANDLE, path=fname), True
    except Exception as exc:
        raise RuntimeError(
            f"Could not download {fname} from Kaggle ({exc!r}).\nAdd Colab secrets KAGGLE_USERNAME / KAGGLE_KEY "
            "(kaggle.com > Settings > API > Create New Token), run kagglehub.login(), or set DATA_DIR.") from exc


cycles, healthy_w, healthy_x, load_log, seq_parts = [], [], [], [], []
for fi, fname in enumerate(FILES):
    t0 = time.time()
    path, downloaded = get_file(fname)
    t_dl = time.time() - t0
    h5_path, repaired = nc.repair_truncated_h5(path, os.getcwd())   # Kaggle's DS08d-010 is 32 bytes short
    if repaired:
        print(f"NOTE: {fname} is shorter than its HDF5 header records (mirror defect); opened a zero-padded copy")
    ds = fname.replace("N-CMAPSS_", "").replace(".h5", "")
    for split in ("dev", "test"):
        t1 = time.time()
        df = nc.load_h5(h5_path, split)
        if df.attrs.get("dropped_rows"):
            print(f"NOTE: dropped {df.attrs['dropped_rows']} physically impossible rows from {fname} [{split}]")
        agg = nc.aggregate_cycles(df)
        agg["split"], agg["dataset"], agg["unit_local"] = split, ds, agg["unit"]
        agg["unit"] = (fi + 1) * 1000 + agg["unit"]
        cycles.append(agg)
        if USE_CNN:
            sp = nc.downsample_1hz(df, CNN_EVERY)
            sp["unit"] = (fi + 1) * 1000 + sp["unit"]
            seq_parts.append(sp)
        if split == "dev":
            h = df[df["hs"] == 1] if "hs" in df else df
            take = rng.choice(len(h), size=min(HEALTHY_SAMPLES_PER_FILE, len(h)), replace=False)
            healthy_w.append(h[nc.W_VARS].to_numpy()[take])
            healthy_x.append(h[nc.XS_VARS].to_numpy()[take])
            del h                                   # copy of up to millions of 1 Hz rows; free before the next file
        load_log.append({"file": fname, "split": split, "rows_1hz": len(df), "engines": agg.unit.nunique(),
                         "cycles": len(agg), "seconds": round(time.time() - t1, 1)})
        print(load_log[-1])
        del df, agg
        gc.collect()
    if repaired:
        os.remove(repaired)
    if downloaded and DELETE_AFTER_USE:
        try:
            os.remove(path)
        except OSError:                             # read-only mount (e.g. /kaggle/input on Kaggle): nothing to free
            pass
    print(f"{fname}: download {t_dl:.0f}s, total {time.time() - t0:.0f}s")
cyc = pd.concat(cycles, ignore_index=True).sort_values(["unit", "cycle"]).reset_index(drop=True)
del cycles
samples = None
if USE_CNN:     # multi-column sort is a stable lexsort -> time order within each flight is kept
    samples = pd.concat(seq_parts, ignore_index=True).sort_values(["unit", "cycle"]).reset_index(drop=True)
del seq_parts
if USE_CNN:
    print(f"{len(samples):,} in-flight samples at {1 / CNN_EVERY:.1f} Hz for the CNN "
          f"({samples.memory_usage().sum() / 1e9:.2f} GB)")
assert not (set(cyc[cyc.split == "dev"].unit) & set(cyc[cyc.split == "test"].unit)), "dev/test engine overlap"
TIMINGS["download_and_aggregate_s"] = time.time() - T_START
print(f"{cyc.unit.nunique()} engines, {len(cyc)} flight cycles "
      f"(train engines {cyc[cyc.split == 'dev'].unit.nunique()}, test engines {cyc[cyc.split == 'test'].unit.nunique()})")

## 4. Layer 2: healthy-engine map ("ideal" sensor values) and residuals

In [ ]:
t0 = time.time()
ref = nc.HealthyReference().fit(np.vstack(healthy_w), np.vstack(healthy_x))
del healthy_w, healthy_x
print("healthy-map relative RMSE per sensor (1 Hz, healthy data):")
print(pd.Series(ref.fit_rmse, index=nc.XS_VARS).map(lambda v: f"{100 * v:.2f}%").to_string())
cyc = nc.add_residuals(cyc, ref)
train_m, test_m = (cyc.split == "dev").to_numpy(), (cyc.split == "test").to_numpy()

# Labels: component health indices from the true health parameters (efficiency/flow modifiers)
D_REF = nc.fit_damage_reference(cyc[train_m])
cyc = pd.concat([cyc, nc.component_health(cyc, D_REF)], axis=1)
label_summary = {}
for c in nc.COMPONENTS:
    hi = cyc.loc[train_m, f"hi_{c}"]
    label_summary[c] = {"min": float(hi.min()), "mean": float(hi.mean()), "std": float(hi.std())}
print("failure-level damage per component D_REF =", {c: round(v, 5) for c, v in D_REF.items()})
print(pd.DataFrame(label_summary).T.round(3))
for c, s in label_summary.items():
    if s["std"] < 0.02:
        print(f"NOTE: {c} health barely varies in the training data - its model can only learn a near-constant value. "
              "Use RUN='full' so every component has abnormal degradation somewhere.")
TIMINGS["layer2_s"] = time.time() - t0

## 5. Layer 3: Kalman filter on the linearised gas-path model (component health parameters + uncertainty)

In [ ]:
t0 = time.time()
kf = nc.GasPathKF().fit(cyc[train_m])
kf_out = kf.run(cyc)
cyc = pd.concat([cyc, kf_out, nc.kf_component_health(kf_out, D_REF)], axis=1)
kf_quality = {}
for c in nc.COMPONENTS:
    t = cyc.loc[test_m, f"hi_{c}"]
    e = cyc.loc[test_m, f"kf_hi_{c}"]
    r = t.corr(e)                                   # NaN when the component never degrades (constant truth)
    kf_quality[c] = {"mae": float((t - e).abs().mean()), "corr": float(r) if np.isfinite(r) else None}
print("Kalman-filter (physics only) component health vs truth on TEST engines:")
print(pd.DataFrame(kf_quality).T.round(3))
TIMINGS["layer3_s"] = time.time() - t0

## 6. Layer 4 inputs: causal per-engine features

In [ ]:
feat = nc.CausalFeaturizer(baseline_cycles=BASELINE_CYCLES, roll=ROLL).fit(cyc[train_m])
frame = feat.transform(cyc)                              # same row order as cyc (sorted by unit, cycle)
assert (frame["unit"].to_numpy() == cyc["unit"].to_numpy()).all()
FEATURES = feat.feature_names
TARGETS = ["rul"] + [f"hi_{c}" for c in nc.COMPONENTS]
SCALE = np.array([RUL_SCALE] + [1.0] * len(nc.COMPONENTS))
X = frame[FEATURES]
Y = cyc[TARGETS].to_numpy(dtype=np.float64)
groups = cyc["unit"].to_numpy()
tr_idx, te_idx = np.flatnonzero(train_m), np.flatnonzero(test_m)
n_folds = min(N_FOLDS, len(np.unique(groups[tr_idx])))
print(f"{len(FEATURES)} tabular features, {len(tr_idx)} train / {len(te_idx)} test cycles, {n_folds}-fold GroupKFold")
if USE_CNN:     # 0.1 Hz windows: flight condition + sensors + residual vs healthy map + this cycle's KF states
    t0 = time.time()
    enc = nc.SampleEncoder()
    s_raw, s_units, s_rows = enc.raw(samples, cyc, ref)
    enc.fit(s_raw, train_m[s_rows])                      # z-score statistics from training engines only
    sbig, sstarts = nc.build_windows_array(enc.normalize(s_raw), s_units, CNN_WINDOW)
    del s_raw
    gc.collect()
    print(f"CNN inputs: {len(enc.channels)} channels x {CNN_WINDOW}-sample windows, {len(s_rows):,} windows "
          f"({time.time() - t0:.0f}s)")

## 7. Tree models per target with out-of-fold predictions (split by engine)

In [ ]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GroupKFold

MODELS = {
    "xgboost": lambda: xgb.XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=5, subsample=0.8,
                                        colsample_bytree=0.6, tree_method="hist", random_state=SEED, n_jobs=-1),
    "lightgbm": lambda: lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10,
                                          subsample=0.8, subsample_freq=1, colsample_bytree=0.6, random_state=SEED,
                                          verbose=-1, n_jobs=-1),
    "catboost": lambda: CatBoostRegressor(iterations=500, learning_rate=0.05, depth=5, random_seed=SEED, verbose=0,
                                          thread_count=-1),
    "randomforest": lambda: RandomForestRegressor(n_estimators=150, max_depth=12, min_samples_leaf=3, max_features=0.4,
                                                  random_state=SEED, n_jobs=-1),
}
TREES = list(MODELS)
BASES = TREES + (["cnn"] if USE_CNN else [])
folds = list(GroupKFold(n_folds).split(tr_idx, groups=groups[tr_idx]))
oof = np.zeros((len(tr_idx), len(TARGETS), len(BASES)))
final = {t: {} for t in TARGETS}
t0 = time.time()
Xtr = X.iloc[tr_idx]


class ConstPred:
    """Used when a target does not vary (a component that never degrades in these engines/folds)."""

    def __init__(self, v):
        self.v = float(v)

    def predict(self, x):
        return np.full(len(x), self.v)


def fit_model(name, x, y):
    if np.ptp(y) < 1e-9:              # CatBoost refuses constant targets; a constant is the honest prediction
        return ConstPred(y[0])
    return MODELS[name]().fit(x, y)


CONST = {tgt: float(Y[tr_idx, ti][0]) for ti, tgt in enumerate(TARGETS) if np.ptp(Y[tr_idx, ti]) < 1e-9}
if CONST:
    print("Constant in the training data (component never degrades in these files) -> predicted as a constant:",
          CONST, "- use RUN='full' so these components have degradation to learn from.")
for ti, tgt in enumerate(TARGETS):
    y = Y[tr_idx, ti]
    if tgt in CONST:
        oof[:, ti, :] = CONST[tgt]
        continue
    for bi, name in enumerate(TREES):
        for a, b in folds:
            m = fit_model(name, Xtr.iloc[a], y[a])
            oof[b, ti, bi] = m.predict(Xtr.iloc[b])
        final[tgt][name] = fit_model(name, Xtr, y)
    oof_rmse = {n: round(float(np.sqrt(np.mean((oof[:, ti, k] - y) ** 2))), 4) for k, n in enumerate(TREES)}
    print(f"{tgt:7s} OOF RMSE", oof_rmse)
TIMINGS["trees_s"] = time.time() - t0

## 8. 1D-CNN on 0.1 Hz in-flight data (multi-output: RUL + 5 component health indices)
Hybrid design of Arias Chao et al. (RESS 2022): 500 s windows of flight condition + sensors, enriched with the
residuals vs the healthy-engine map and the Kalman-filter health parameters. Per-cycle prediction = mean over the
windows ending in that flight.

In [ ]:
cnn_final = cnn_cycles_all = None
TIMINGS["cnn_s"] = 0.0
if USE_CNN:
    import torch
    import torch.nn as nn

    DEVICE = "cpu"
    if torch.cuda.is_available():
        try:    # recent PyTorch builds have no kernels for old GPUs (e.g. Kaggle P100): probe before training
            nn.Conv1d(2, 2, 3).cuda()(torch.zeros(1, 2, 8, device="cuda")).sum().item()
            DEVICE = "cuda"
        except Exception as exc:
            print("CUDA GPU not usable with this PyTorch build, training on CPU:", repr(exc))
    Ys = torch.tensor(Y / SCALE, dtype=torch.float32)

    class CNNNet(nn.Module):
        """3 conv layers (kernel 10, 'same' zero padding, 10/10/1 channels), flatten, dense 50, linear multi-output
        head. Input (batch, window, channels)."""

        def __init__(self, n_in, n_out, window=CNN_WINDOW):
            super().__init__()
            layers, chans = [], [n_in, 10, 10, 1]
            for a, b in zip(chans[:-1], chans[1:]):
                layers += [nn.ConstantPad1d((4, 5), 0.0), nn.Conv1d(a, b, 10), nn.ReLU()]
            self.conv = nn.Sequential(*layers)
            self.head = nn.Sequential(nn.Flatten(), nn.Linear(window, 50), nn.ReLU(), nn.Linear(50, n_out))

        def forward(self, x):
            return self.head(self.conv(x.transpose(1, 2)))

    def train_cnn(cycle_rows_allowed, seed=SEED):
        torch.manual_seed(seed)
        r = np.random.default_rng(seed)
        idx = np.flatnonzero(np.isin(s_rows, cycle_rows_allowed))
        model = CNNNet(len(enc.channels), len(TARGETS)).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=CNN_LR, amsgrad=True)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CNN_EPOCHS)
        for _ in range(CNN_EPOCHS):
            model.train()
            perm = r.choice(idx, size=min(len(idx), CNN_SAMPLES_PER_EPOCH), replace=False)
            for i in range(0, len(perm), CNN_BATCH):
                bt = perm[i:i + CNN_BATCH]
                xb = torch.from_numpy(nc.gather_windows(sbig, sstarts, CNN_WINDOW, bt)).to(DEVICE)
                loss = nn.functional.mse_loss(model(xb), Ys[torch.from_numpy(s_rows[bt])].to(DEVICE))
                opt.zero_grad()
                loss.backward()
                opt.step()
            sched.step()
        return model.eval()

    def predict_cnn_cycles(model, cycle_rows_wanted):
        """Per-cycle CNN outputs in target units, shape (len(cyc), n_targets); NaN for cycles not requested."""
        idx = np.flatnonzero(np.isin(s_rows, cycle_rows_wanted))
        out = []
        with torch.no_grad():
            for i in range(0, len(idx), 8192):
                xb = torch.from_numpy(nc.gather_windows(sbig, sstarts, CNN_WINDOW, idx[i:i + 8192])).to(DEVICE)
                out.append(model(xb).cpu().numpy() * SCALE)
        return nc.cycle_mean(s_rows[idx], np.concatenate(out), len(cyc))

    t0 = time.time()
    ci = BASES.index("cnn")
    for k, (a, b) in enumerate(folds):
        m = train_cnn(tr_idx[a], seed=SEED + k)
        oof[b, :, ci] = predict_cnn_cycles(m, tr_idx[b])[tr_idx[b]]
    cnn_final = train_cnn(tr_idx)
    cnn_cycles_all = predict_cnn_cycles(cnn_final, np.arange(len(cyc)))
    TIMINGS["cnn_s"] = time.time() - t0
    print("CNN OOF RMSE per target:",
          {t: round(float(np.sqrt(np.mean((oof[:, k, ci] - Y[tr_idx, k]) ** 2))), 4) for k, t in enumerate(TARGETS)},
          f"({TIMINGS['cnn_s']:.0f}s on {DEVICE})")

## 9. Per target: non-negative Ridge blend OR the best single model (chosen by cross-validation on training engines), then causal smoothing

In [ ]:
class Blend:
    """Linear blend of base predictions: Ridge weights, or one-hot weights = a single model."""

    def __init__(self, coef, intercept):
        self.coef_, self.intercept_ = np.asarray(coef, dtype=np.float64), float(intercept)

    def predict(self, b):
        return b @ self.coef_ + self.intercept_


def clip_target(tgt, v):
    return nc.clip_target(tgt, v)


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


def ridge_for(tgt):
    return Ridge(alpha=1.0 if tgt == "rul" else 1e-3, positive=True)


def choose_blend(ti, tgt, cols):
    """Blend of base columns `cols` if its cross-validated RMSE beats the best single model by STACK_MIN_GAIN,
    otherwise that single model. Returns (Blend, choice, CV predictions on training rows, report)."""
    y = Y[tr_idx, ti]
    cv = np.zeros(len(tr_idx))
    for a, b in folds:      # meta-model fitted without the fold it predicts -> honest blend RMSE
        cv[b] = ridge_for(tgt).fit(oof[a][:, ti, cols], y[a]).predict(oof[b][:, ti, cols])
    cv = clip_target(tgt, cv)
    singles = {BASES[c]: rmse(clip_target(tgt, oof[:, ti, c]), y) for c in cols}
    best = min(singles, key=singles.get)
    rep = {"blend_cv_rmse": rmse(cv, y), "single_cv_rmse": singles}
    if rep["blend_cv_rmse"] <= (1 - STACK_MIN_GAIN) * singles[best]:
        m = ridge_for(tgt).fit(oof[:, ti, cols], y)
        return Blend(m.coef_, m.intercept_), "blend", cv, rep
    coef = np.zeros(len(cols))
    coef[cols.index(BASES.index(best))] = 1.0
    return Blend(coef, 0.0), best, clip_target(tgt, oof[:, ti, BASES.index(best)]), rep


meta, meta_trees, model_choice, SMOOTH = {}, {}, {}, {}
tr_units, tr_cycles = groups[tr_idx], cyc["cycle"].to_numpy()[tr_idx]
for ti, tgt in enumerate(TARGETS):
    if tgt in CONST:
        meta[tgt] = meta_trees[tgt] = None
        SMOOTH[tgt] = 1.0
        print(f"{tgt:7s} constant = {CONST[tgt]}")
        continue
    meta_trees[tgt], tchoice, _, _ = choose_blend(ti, tgt, list(range(len(TREES))))
    meta[tgt], choice, cv, rep = choose_blend(ti, tgt, list(range(len(BASES))))
    y = Y[tr_idx, ti]
    sm = {a: rmse(nc.smooth_target(tgt, tr_units, tr_cycles, cv, a), y) for a in SMOOTH_GRID}
    SMOOTH[tgt] = min(sm, key=sm.get)
    model_choice[tgt] = {"full": choice, "trees_only": tchoice, **rep, "smoothing_alpha": SMOOTH[tgt],
                         "smoothed_cv_rmse": sm[SMOOTH[tgt]]}
    best = min(rep["single_cv_rmse"], key=rep["single_cv_rmse"].get)
    print(f"{tgt:7s} -> {choice:12s} (blend CV RMSE {rep['blend_cv_rmse']:.4f}, best single {best} "
          f"{rep['single_cv_rmse'][best]:.4f}); smoothing alpha {SMOOTH[tgt]} -> CV RMSE {sm[SMOOTH[tgt]]:.4f}; "
          f"weights {dict(zip(BASES, np.round(meta[tgt].coef_, 3)))}")


def base_predictions(rows):
    """(len(rows), n_targets, n_bases) predictions of every base model."""
    out = np.zeros((len(rows), len(TARGETS), len(BASES)))
    xr = X.iloc[rows]
    for ti, tgt in enumerate(TARGETS):
        if tgt in CONST:
            out[:, ti, :] = CONST[tgt]
            continue
        for bi, name in enumerate(TREES):
            out[:, ti, bi] = final[tgt][name].predict(xr)
    if USE_CNN:
        out[:, :, BASES.index("cnn")] = cnn_cycles_all[rows]
    return out


def stack(base, rows, trees_only=False, smooth=True):
    """Blend (or chosen single model) per target, clipped and causally smoothed per engine (rows sorted)."""
    res = np.zeros(base.shape[:2])
    units, cycles = groups[rows], cyc["cycle"].to_numpy()[rows]
    for ti, tgt in enumerate(TARGETS):
        if tgt in CONST:
            res[:, ti] = CONST[tgt]
            continue
        m = meta_trees[tgt] if trees_only else meta[tgt]
        v = m.predict(base[:, ti, :len(TREES)] if trees_only else base[:, ti, :])
        res[:, ti] = nc.smooth_target(tgt, units, cycles, v, SMOOTH[tgt] if smooth else 1.0)
    return res

## 10. Evaluation on held-out TEST engines (all cycles) + baselines + ablation

In [ ]:
def nasa_score(pred, true):
    d = np.asarray(pred) - np.asarray(true)
    return np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)


def report(pred, true, rul=False):
    d = np.asarray(pred) - np.asarray(true)
    r = {"rmse": float(np.sqrt(np.mean(d ** 2))), "mae": float(np.mean(np.abs(d)))}
    if rul:
        s = nasa_score(pred, true)
        r.update({"nasa_score_sum": float(s.sum()), "nasa_score_mean": float(s.mean())})
    return r


base_te = base_predictions(te_idx)
pred_te, pred_te_trees = stack(base_te, te_idx), stack(base_te, te_idx, trees_only=True)
pred_te_unsmoothed = stack(base_te, te_idx, smooth=False)
metrics = {"test_engines": int(len(np.unique(groups[te_idx]))), "test_cycles": int(len(te_idx))}
rows = []
for ti, tgt in enumerate(TARGETS):
    truth = Y[te_idx, ti]
    is_rul = tgt == "rul"
    metrics[f"{tgt}_stack"] = report(pred_te[:, ti], truth, is_rul)
    metrics[f"{tgt}_trees_only"] = report(pred_te_trees[:, ti], truth, is_rul)
    metrics[f"{tgt}_stack_unsmoothed"] = report(pred_te_unsmoothed[:, ti], truth, is_rul)
    deg = truth <= 20 if is_rul else truth < 0.9          # where it matters: end of life / degraded component
    if deg.sum() >= 5:
        metrics[f"{tgt}_stack_degraded"] = report(pred_te[deg, ti], truth[deg], is_rul)
        if not is_rul:
            metrics[f"{tgt}_kalman_filter_only_degraded"] = report(cyc[f"kf_{tgt}"].to_numpy()[te_idx][deg], truth[deg])
    for bi, name in enumerate(BASES):
        metrics[f"{tgt}_{name}"] = report(clip_target(tgt, base_te[:, ti, bi]), truth, is_rul)
    if not is_rul:
        metrics[f"{tgt}_kalman_filter_only"] = report(cyc[f"kf_{tgt}"].to_numpy()[te_idx], truth)
    rows.append({"target": tgt, "chosen": model_choice.get(tgt, {}).get("full", "constant"),
                 "stack": metrics[f"{tgt}_stack"]["rmse"], "unsmoothed": metrics[f"{tgt}_stack_unsmoothed"]["rmse"],
                 "degraded_only": metrics.get(f"{tgt}_stack_degraded", {}).get("rmse"),
                 "trees_only": metrics[f"{tgt}_trees_only"]["rmse"],
                 "physics_KF_only": metrics.get(f"{tgt}_kalman_filter_only", {}).get("rmse")})
# naive baseline the spec asks to beat: linear regression on raw per-cycle sensor means
raw_cols = [f"m_{s}" for s in nc.XS_VARS] + nc.FLIGHT_COLS
lin = LinearRegression().fit(cyc.loc[train_m, raw_cols], Y[tr_idx, 0])
metrics["rul_naive_linear_raw_sensors"] = report(np.clip(lin.predict(cyc.loc[test_m, raw_cols]), 0, None), Y[te_idx, 0], True)
ablation = pd.DataFrame(rows).set_index("target")
print("Test RMSE (RUL in cycles; health indices on 0-1 scale, x100 = health points; "
      "degraded_only = RUL <= 20 / health < 0.9):")
print(ablation.round(4))
print("RUL naive linear baseline RMSE:", round(metrics["rul_naive_linear_raw_sensors"]["rmse"], 2))
if USE_CNN:
    print("'stack' vs 'trees_only' shows what the CNN adds (it costs interpretability).")

## 11. Uncertainty: RUL quantile models (90% interval)

In [ ]:
q_models = {q: lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=400, learning_rate=0.03, num_leaves=15,
                                 min_child_samples=10, subsample=0.8, subsample_freq=1, colsample_bytree=0.6,
                                 random_state=SEED, verbose=-1, n_jobs=-1).fit(Xtr, Y[tr_idx, 0])
            for q in (0.05, 0.95)}
lo = np.clip(q_models[0.05].predict(X.iloc[te_idx]), 0, None)
hi = np.clip(q_models[0.95].predict(X.iloc[te_idx]), 0, None)
cover = float(np.mean((Y[te_idx, 0] >= lo) & (Y[te_idx, 0] <= hi)))
metrics["rul_interval90_coverage"] = cover
metrics["rul_interval90_mean_width"] = float(np.mean(hi - lo))
print(f"RUL 90% interval coverage on test engines: {cover:.1%} (target 90%), mean width {np.mean(hi - lo):.1f} cycles")

## 12. Explainability: TreeSHAP per target (weighted by the Ridge blend weights), grouped by physical source

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

t0 = time.time()
sample = Xtr.sample(min(1500, len(Xtr)), random_state=SEED)
shap_rows = []
for tgt in TARGETS:
    if tgt in CONST:
        continue
    total = np.zeros(sample.shape)
    for bi, name in enumerate(TREES):
        w = float(meta[tgt].coef_[bi])
        if w == 0:
            continue
        sv = shap.TreeExplainer(final[tgt][name]).shap_values(sample)
        total += w * np.asarray(sv).reshape(sample.shape)
    imp = np.abs(total).mean(0)
    for f_, v in zip(FEATURES, imp):
        shap_rows.append({"target": tgt, "feature": f_, "group": nc.source_of(f_), "mean_abs_shap": float(v)})
shap_df = pd.DataFrame(shap_rows)
shap_df.to_csv("shap_importance.csv", index=False)
by_group = shap_df.groupby(["target", "group"])["mean_abs_shap"].sum().reset_index()
by_group.to_csv("shap_importance_by_group.csv", index=False)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, tgt in zip(axes.ravel(), TARGETS):
    if tgt in CONST:
        ax.set_title(f"{tgt}: constant in training data")
        ax.axis("off")
        continue
    top = by_group[by_group.target == tgt].nlargest(8, "mean_abs_shap")[::-1]
    ax.barh(top["group"], top["mean_abs_shap"], color="#0284c7")
    ax.set_title(f"{tgt}: top drivers (tree part, |SHAP|)")
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=130)
TIMINGS["shap_s"] = time.time() - t0
print(by_group.sort_values("mean_abs_shap", ascending=False).groupby("target").head(3).to_string(index=False))

## 13. Cost: timings and inference latency

In [ ]:
one = np.flatnonzero(groups == groups[te_idx[0]])                  # all cycles of one test engine
t0 = time.time()
for _ in range(5):
    stack(base_predictions(one), one)
metrics["inference_ms_one_engine_all_cycles"] = (time.time() - t0) / 5 * 1000
TIMINGS["total_s"] = time.time() - T_START
metrics["timings_s"] = {k: round(v, 1) for k, v in TIMINGS.items()}
print(json.dumps(metrics["timings_s"], indent=1), f"\none-engine inference {metrics['inference_ms_one_engine_all_cycles']:.0f} ms")

## 14. Export the bundle for the backend (native model formats + JSON) and download

In [ ]:
import copy

import joblib

OUT = f"bundle_{RUN}"
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT)
EXT = {"xgboost": "json", "lightgbm": "txt", "catboost": "cbm", "randomforest": "joblib"}
targets_manifest = {}
for ti, tgt in enumerate(TARGETS):
    if tgt in CONST:
        cm = {"coef": [], "intercept": CONST[tgt]}
        targets_manifest[tgt] = {"constant": CONST[tgt], "base_models": [], "meta": cm, "trees_only_meta": cm}
        continue
    bases = []
    for name in TREES:
        fn = f"{name}_{tgt}.{EXT[name]}"
        mdl = final[tgt][name]
        if name == "xgboost":
            mdl.save_model(os.path.join(OUT, fn))
        elif name == "lightgbm":
            mdl.booster_.save_model(os.path.join(OUT, fn))
        elif name == "catboost":
            mdl.save_model(os.path.join(OUT, fn))
        else:
            joblib.dump(mdl, os.path.join(OUT, fn), compress=3)
        bases.append({"type": name, "file": fn})
    if USE_CNN:
        bases.append({"type": "cnn", "output_index": ti})
    targets_manifest[tgt] = {
        "base_models": bases,
        "meta": {"coef": [float(c) for c in meta[tgt].coef_], "intercept": float(meta[tgt].intercept_)},
        "trees_only_meta": {"coef": [float(c) for c in meta_trees[tgt].coef_],
                            "intercept": float(meta_trees[tgt].intercept_)},
    }
for q, m in q_models.items():
    m.booster_.save_model(os.path.join(OUT, f"lightgbm_rul_q{int(q * 100):02d}.txt"))
json.dump(ref.to_dict(), open(os.path.join(OUT, "reference.json"), "w"))
json.dump(kf.to_dict(), open(os.path.join(OUT, "kf.json"), "w"))
json.dump(feat.to_dict(), open(os.path.join(OUT, "featurizer.json"), "w"))
for f_ in ("shap_importance.csv", "shap_importance_by_group.csv", "shap_summary.png"):
    shutil.copy(f_, OUT)

cnn_manifest = None
if USE_CNN:
    torch.save(cnn_final.state_dict(), os.path.join(OUT, "cnn.pt"))
    cnn_manifest = {"file": None, "weights_file": "cnn.pt", "window": CNN_WINDOW, "every": CNN_EVERY,
                    "encoder": enc.to_dict(), "outputs": TARGETS, "output_scale": SCALE.tolist(),
                    "padding": "repeat_first_sample"}
    try:
        class Scaled(nn.Module):
            def __init__(self, m):
                super().__init__()
                self.m = m
                self.register_buffer("scale", torch.tensor(SCALE, dtype=torch.float32))

            def forward(self, x):
                return self.m(x) * self.scale

        cpu_model = Scaled(copy.deepcopy(cnn_final).cpu().eval()).eval()
        kw = dict(input_names=["window"], output_names=["targets"],
                  dynamic_axes={"window": {0: "batch"}, "targets": {0: "batch"}}, opset_version=17)
        onnx_path = os.path.join(OUT, "cnn.onnx")
        example = torch.zeros(1, CNN_WINDOW, len(enc.channels))
        try:
            torch.onnx.export(cpu_model, example, onnx_path, dynamo=False, **kw)
        except TypeError:
            torch.onnx.export(cpu_model, example, onnx_path, **kw)
        import onnxruntime as ort
        wins = nc.gather_windows(sbig, sstarts, CNN_WINDOW, np.flatnonzero(np.isin(s_rows, te_idx))[:256])
        with torch.no_grad():
            ref_out = cpu_model(torch.from_numpy(wins)).numpy()
        got = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"]).run(None, {"window": wins})[0]
        diff = float(np.max(np.abs(ref_out - got)))
        metrics["cnn_onnx_max_abs_diff"] = diff
        if diff < 1e-3:
            cnn_manifest["file"] = "cnn.onnx"
        print(f"CNN ONNX export: max |torch - onnx| = {diff:.2e}")
    except Exception as exc:
        print("WARNING: CNN ONNX export failed; the backend will fall back to the tree-only blend:", repr(exc))
    # raw 0.1 Hz samples of the test engines: the backend recomputes residuals, KF states and CNN inputs itself
    ds = samples[np.isin(samples["unit"].to_numpy(), np.unique(groups[te_idx]))]
    np.savez_compressed(os.path.join(OUT, "demo_samples.npz"), unit=ds["unit"].to_numpy(np.int32),
                        cycle=ds["cycle"].to_numpy(np.int32), values=ds[nc.W_VARS + nc.XS_VARS].to_numpy(np.float32),
                        columns=np.array(nc.W_VARS + nc.XS_VARS))

# Demo data for the dashboard = the held-out TEST engines (per-cycle aggregates; the backend recomputes Layers 2-4)
demo_cols = ["unit", "unit_local", "dataset", "cycle", "fc", "hs", "rul", "duration_s", "alt_max", "Mach_max"] + \
    [f"{v}_mean" for v in nc.W_VARS] + [c for c in cyc.columns if c.startswith(("m_", "sd_", "p_", "v_", "phi_"))] + \
    nc.T_VARS + [f"hi_{c}" for c in nc.COMPONENTS]
cyc.loc[test_m, [c for c in demo_cols if c in cyc.columns]].to_csv(os.path.join(OUT, "demo_cycles.csv"), index=False)
par = cyc.loc[test_m, ["unit", "cycle"]].copy()
for ti, tgt in enumerate(TARGETS):
    par[f"pred_{tgt}"] = pred_te[:, ti]
    par[f"pred_{tgt}_trees_only"] = pred_te_trees[:, ti]
for c in nc.COMPONENTS:
    par[f"kf_hi_{c}"] = cyc.loc[test_m, f"kf_hi_{c}"].to_numpy()
par["rul_q05"], par["rul_q95"] = lo, hi
par.to_csv(os.path.join(OUT, "parity_outputs.csv"), index=False)

manifest = {
    "version": 3, "dataset": "NASA N-CMAPSS", "run": RUN, "files": FILES, "components": nc.COMPONENTS,
    "targets": targets_manifest, "target_order": TARGETS, "rul_scale": RUL_SCALE, "d_ref": D_REF,
    "band_edges": BAND_EDGES, "features": FEATURES, "featurizer_file": "featurizer.json",
    "reference_file": "reference.json", "kf_file": "kf.json", "cnn": cnn_manifest,
    "uncertainty": {"rul_q05_file": "lightgbm_rul_q05.txt", "rul_q95_file": "lightgbm_rul_q95.txt",
                    "coverage_test": cover},
    "demo_file": "demo_cycles.csv", "demo_samples_file": "demo_samples.npz" if USE_CNN else None,
    "parity_file": "parity_outputs.csv", "model_choice": model_choice, "smoothing": SMOOTH,
    "metrics": metrics, "kf_quality_test": kf_quality, "label_summary": label_summary, "load_log": load_log,
    "notes": ["Health index 1 = healthy, 0 = failure-level damage of THAT component (d_ref per component).",
              "Predictions are causally smoothed per engine (alpha per target in 'smoothing'; RUL tracks -1/cycle).",
              "Virtual sensors (v_*) are display-only; models use W + X_s derived features only."],
}
json.dump(manifest, open(os.path.join(OUT, "manifest.json"), "w"), indent=2)
import catboost
import sklearn
versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "xgboost": xgb.__version__, "lightgbm": lgb.__version__, "catboost": catboost.__version__,
            "scikit-learn": sklearn.__version__, "shap": shap.__version__, "joblib": joblib.__version__}
if USE_CNN:
    versions["torch"] = torch.__version__
json.dump(versions, open(os.path.join(OUT, "versions.json"), "w"), indent=2)
json.dump(metrics, open(os.path.join(OUT, "metrics.json"), "w"), indent=2)
zip_path = shutil.make_archive(f"genzone_ncmapss_{RUN}_bundle", "zip", OUT)
print("Bundle:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB); total time {(time.time() - T_START) / 60:.1f} min")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)